# Run RWE on Reddit Politosphere (behavioral ideological axis)

Unlike the MIND notebook (where the left–right axis is a *text* proxy scored from
headlines — a weak, topic-vs-stance-conflated signal), this runs RWE on the
**Reddit Politosphere** and learns the ideological axis from **behavior**: who
participates in which political subreddit. That's the ideal-point method
(`rwe.IdeologyModel`) — a *validated* ideology measure — so RQ3's bridging is on a
**genuine** axis, with a clean `lean_corr` validation number (no Twitter API).

Pipeline: download a slice → ingest to a user×subreddit `.npz` → fit the ideal-point
axis (oriented to known subreddit leans) → RQ2/RQ3 + the axis plot. Everything is
cached to Drive, so it survives runtime resets.

> **License:** confirm the terms on <https://zenodo.org/records/5851729> before use.
> Politosphere is pseudonymized and derived from Pushshift; nothing is committed.

In [ ]:
# 1) Get the code (branch with the Politosphere pipeline) and install it
import os
if not os.path.isdir("/content/random_walks_with_erasure"):
    get_ipython().system("git clone --branch claude/sleepy-gates-oecof1 https://github.com/greenwichg/random_walks_with_erasure.git /content/random_walks_with_erasure")
os.chdir("/content/random_walks_with_erasure")
get_ipython().system("pip install -e . -q")
print("installed ->", os.getcwd())

In [ ]:
# 1b) Drive cache — make every expensive artifact (the comment files, the .npz)
#     survive Colab runtime resets. Mounts Drive once; later cells call
#     cache_get / cache_put, so after the first run you never re-download.
import os, shutil

CACHE = "/content/drive/MyDrive/rwe_polito"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CACHE, exist_ok=True)
    CACHE_OK = True
    print("Drive cache ready ->", CACHE)
except Exception as e:
    CACHE_OK = False
    print("(no Drive cache; artifacts will NOT persist across resets):", e)

def cache_get(name):
    """Copy <name> back from the Drive cache into the working dir if available."""
    src = os.path.join(CACHE, os.path.basename(name))
    if CACHE_OK and os.path.exists(src) and not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        shutil.copy(src, name)
        print("restored from Drive cache:", name)
    return os.path.exists(name)

def cache_put(name):
    """Save <name> to the Drive cache for future runs."""
    if CACHE_OK and os.path.exists(name):
        shutil.copy(name, os.path.join(CACHE, os.path.basename(name)))
        print("cached to Drive:", name)

In [ ]:
# 2) Download a slice of Politosphere from Zenodo (record 5851729). Default: the
#    US-2016-election window (matches the original RWE paper's 'US elections 2016').
#    Cached to Drive -> reset-safe. Add/extend MONTHS for more data (bigger = slower).
import os, glob
os.makedirs("politosphere", exist_ok=True)
MONTHS = ["2016-09", "2016-10", "2016-11"]
BASE = "https://zenodo.org/records/5851729/files"
for m in MONTHS:
    path = f"politosphere/comments_{m}.bz2"
    if cache_get(path):
        continue
    print("downloading", os.path.basename(path), "...")
    get_ipython().system(f"wget -q -O {path} '{BASE}/comments_{m}.bz2?download=1'")
    ok = (os.path.exists(path) and os.path.getsize(path) > 10000
          and open(path, "rb").read(3) == b"BZh")          # real bzip2, not an HTML 404
    if ok:
        cache_put(path)
    else:
        if os.path.exists(path):
            os.remove(path)
        print(f"  !! comments_{m}.bz2 did not download as a .bz2. Check the exact file "
              "name in the Files section of https://zenodo.org/records/5851729 (it may "
              "be bundled differently), or download it manually and drop it into the "
              "politosphere/ folder, then re-run.")
print("have:", sorted(glob.glob("politosphere/*.bz2")))

In [ ]:
# 3) Ingest -> user×subreddit matrix + ideal-point ideology axis (oriented to the
#    bundled subreddit leans). Cached to Drive (reset-safe).
if not cache_get("politosphere.npz"):
    get_ipython().system("python examples/ingest_politosphere.py --comments-dir politosphere "
                         "--ideology --min-user-clicks 5 --min-item-clicks 20 "
                         "--sample-users 15000 --out politosphere.npz")
    cache_put("politosphere.npz")
# WATCH the printed lean_corr: near +-1 = the behavioural axis matches the known
# subreddit leans -- the validation the headline axis (~0) could never give.

In [ ]:
# 4) Evaluate: baselines + RWE-D/RWE-B -> RQ2 (long-tail) + RQ3 (ideological
#    bridging) on the BEHAVIOURAL axis. Same driver as MIND -- the .npz is drop-in.
get_ipython().system("python examples/eval_mind.py --npz politosphere.npz --no-bprmf "
                     "--out-csv politosphere_results.csv")
import pandas as pd
print("\nRESULTS (Politosphere, ideal-point axis):")
print(pd.read_csv("politosphere_results.csv", index_col=0).round(3).to_string())

In [ ]:
# 5) Where users + subreddits sit on the learned left<->right axis
get_ipython().system("python examples/plot_axis.py --npz politosphere.npz --out polito_axis.png")
from IPython.display import Image, display
display(Image("polito_axis.png"))
try:
    from google.colab import files; files.download("polito_axis.png")
except Exception:
    pass

## What to look at

- **`lean_corr`** (cell 3) — the headline number. Near ±1 means the behavioral
  axis lines up with the known subreddit leans: a *clean* ideological axis, vs the
  MIND headline proxy's ~0. This is the whole point of the pivot.
- **RQ3 table** (cell 4) — `uw_shift` / `uw_recs` for RWE-B vs the baselines, now on a
  genuine ideological axis. RWE-B should still bridge hardest.
- **The axis plot** (cell 5) — left-leaning subreddits on the left, right on the right,
  users spread between them.

Paste `lean_corr` + the RQ2/RQ3 table back, and it gets folded into `RESULTS.md` /
the paper as a third dataset — the one that finally gives RQ3 a validated axis.

**Scale:** start with a few months; the de-dup grows with #users, and the ideal-point
fit is dense `O(users×items)` (capped by `--sample-users`). Extend `MONTHS` once a
small run works end-to-end.